In [26]:
## BIBLIOTECAS UTILIZADAS

import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

# Baixar recursos necessários do NLTK
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuário\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [27]:
# Ler o csv com tabelas e printar ele
csv_path = 'tables.csv'
data = pd.read_csv(csv_path)
print(data.head(10))

  DATABASE   SCHEMA  TABELA        CAMPO
0      db1  schema1  table1           id
1      db1  schema1  table1         name
2      db1  schema1  table2           id
3      db1  schema2  table2  description
4      db1  schema2  table3        price
5      db2  schema1  table1           id
6      db2  schema1  table2         name
7      db2  schema2  table2  description
8      db2  schema2  table3        price
9      db2  schema3  table4     category


In [43]:
# Extrair palavras-chave usando TF-IDF e retorna a matriz TF-IDF

def extract_keywords(texts):
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(texts)
    return X


<function extract_keywords at 0x00000196FFDD7920>


In [37]:
# Cálculo da similaridade de cosseno entre as chaves das tabelas

def calculate_similarity(matrix):
    return cosine_similarity(matrix)

In [38]:
# Comparar tabelas com base nos campos e chave primária

def compare_tables(data, table_col, db_col, schema_col, field_col):
    """Compara tabelas com base nas informações dos campos."""
    # Criar uma coluna combinada para identificação completa
    data['full_table_id'] = data[db_col] + '.' + data[schema_col] + '.' + data[table_col]
    
    # Obter todos os nomes de tabelas únicas
    unique_tables = data['full_table_id'].unique()
    table_texts = {}

    for table_id in unique_tables:
        # Filtrar dados da tabela específica
        table_data = data[data['full_table_id'] == table_id]
        
        # Concatenar o texto de todos os campos
        concatenated_text = ' '.join(table_data[field_col].astype(str).tolist())
        table_texts[table_id] = concatenated_text
    
    # Criar a matriz TF-IDF
    matrix = extract_keywords(list(table_texts.values()))
    
    # Calcular similaridade entre tabelas
    similarity_matrix = calculate_similarity(matrix)
    
    # Criar um DataFrame para visualizar a similaridade
    similarity_df = pd.DataFrame(similarity_matrix, index=table_texts.keys(), columns=table_texts.keys())
    
    return similarity_df

In [41]:
# Definir colunas esperadas no CSV
table_col = 'TABELA'
db_col = 'DATABASE'
schema_col = 'SCHEMA'
field_col = 'CAMPO'

# Caminho para o arquivo CSV de saída
output_csv_path = 'similarity_matrix.csv'

# Comparar tabelas e calcular similaridade
similarity_df = compare_tables(data, table_col, db_col, schema_col, field_col)

# Salvar a matriz de similaridade em um novo arquivo CSV
similarity_df.to_csv(output_csv_path)

print(f"Matriz de Similaridade entre tabelas salva em: {output_csv_path}")

Matriz de Similaridade entre tabelas salva em: similarity_matrix.csv
